# Milestone 3 Lab, notebook 6 of 6: The same question, with real warnings

## Where we are

| Notebook | What you do |
|---|---|
| 1. Where we are, and the problems by hand | Recall the question, find the data problems yourself, and correct the answer key |
| 2. Checks in code | Write two quality checks as queries, and prove each one works |
| 3. Quarantine, and the figures before and after | Set the wrong records aside, recalculate, and confirm the totals add up |
| 4. A to-do list and a helper, by hand | Ask a model to explain the problems, then build an agent that organises its own work |
| 5. A harness works the queue | Give every problem to a ready-made agent harness, and score its reports |
| **6. The same question, with real warnings** (you are here) | Ask the analyst the leader's question, now that the warnings come from real checks |

Model requests in this notebook: **about 8**. A full run of this lab costs about 10 US cents.

## What this notebook is for

Back to the retail leader, who knows nothing about any of this and asks the same question as in milestone 1.

The analyst is almost the same: the same six tools, the same answer format, the same verifier. One thing behind it has changed. The quality tool no longer reads a hand-written list. It reads the **findings** the checks produced.

You will see the warnings change, the confidence change, and one figure change.

In [1]:
import pandas as pd
from lab_helpers import (
    LAST_MONTH,
    MAIN_QUESTION,
    PREVIOUS_MONTH,
    check_key_against_calculated,
    load_result,
    show_answer,
    show_doc,
)

from retail_profit_stack import tools
from retail_profit_stack.agent import run_analyst
from retail_profit_stack.evaluation import load_cases, run_evaluation, summarise
from retail_profit_stack.store import get_store
from retail_profit_stack.transform.pipeline import run_pipeline

## Step 1: Run the pipeline

One call checks the records, sets the wrong ones aside, calculates the figures, and writes the findings.

In [2]:
run = run_pipeline("sql")
get_store.cache_clear()  # make the tools load the tables the pipeline just wrote

pd.Series(run.check_outcomes, name="outcome").to_frame()

,outcome
duplicate_records,quarantine
missing_product_cost,needs_review
event_out_of_order,needs_review
refund_exceeds_line,needs_review
totals_do_not_match,pass


## Step 2: The quality tool, before and after

This is the list a person wrote by hand, which milestones 1 and 2 used.

In [3]:
pd.DataFrame([
    {"flag": "QF-001", "month": "2026-10", "product": "P007", "problem": "missing product cost", "severity": "high"},
    {"flag": "QF-002", "month": "2026-10", "product": "P001", "problem": "duplicate orders", "severity": "medium"},
    {"flag": "QF-003", "month": "2026-09", "product": "P011", "problem": "refund exceeds order", "severity": "low"},
])

,flag,month,product,problem,severity
0,QF-001,2026-10,P007,missing product cost,high
1,QF-002,2026-10,P001,duplicate orders,medium
2,QF-003,2026-09,P011,refund exceeds order,low


And this is what the same tool, `get_quality_flags`, returns now for October. Its name and arguments have not changed. Only where its data comes from.

In [4]:
october = tools.get_quality_flags(LAST_MONTH)

pd.DataFrame([flag.model_dump() for flag in october.flags])[["flag_id", "product_id", "check_name", "outcome", "severity", "record_count"]]

,flag_id,product_id,check_name,outcome,severity,record_count
0,QF-002,P001,duplicate_records,quarantine,low,9
1,QF-003,P002,duplicate_records,quarantine,low,12
2,QF-004,P007,missing_product_cost,needs_review,high,0
3,QF-006,P003,event_out_of_order,needs_review,low,1


Four findings for October where the list had two. Each one now carries its **outcome**: whether the records were set aside or are still in the figures.

In [5]:
for flag in october.flags:
    print(flag.flag_id, "-", flag.description)

QF-002 - Records for Trail water bottle in 2026-10 that appeared more than once: 9. The extra copies were set aside and are not in the figures.
QF-003 - Records for Insulated flask in 2026-10 that appeared more than once: 12. The extra copies were set aside and are not in the figures.
QF-004 - No product cost is recorded for Camp stove in 2026-10, so its gross profit and contribution profit cannot be calculated. Its category total leaves it out.
QF-006 - Records for Hydration pack in 2026-10 dated before their order: 1. They are still counted, in the month of their own date.


## Step 3: Confidence comes from the checks

The rule is unchanged from milestone 1: the most severe finding in scope sets the confidence. What changed is the findings.

Ask for confidence in three scopes.

In [6]:
whole_month = tools.get_quality_flags(LAST_MONTH)
bottle_only = tools.get_quality_flags(LAST_MONTH, product_id="P001")
september = tools.get_quality_flags(PREVIOUS_MONTH)

pd.DataFrame([
    {"scope": "All of October", "confidence": whole_month.confidence_level, "findings": [flag.flag_id for flag in whole_month.flags]},
    {"scope": "Trail water bottle, October", "confidence": bottle_only.confidence_level, "findings": [flag.flag_id for flag in bottle_only.flags]},
    {"scope": "All of September", "confidence": september.confidence_level, "findings": [flag.flag_id for flag in september.flags]},
])

,scope,confidence,findings
0,All of October,low,"[QF-002, QF-003, QF-004, QF-006]"
1,"Trail water bottle, October",high,[QF-002]
2,All of September,medium,"[QF-001, QF-005, QF-007]"


Look at the trail water bottle. In milestone 2 its confidence was **medium**, because its duplicates were known but still counted. Now it is **high**: the duplicates were found and set aside, so the figure can be trusted. Handling a problem earns back confidence.

October as a whole is still **low**, because the camp stove's cost is still missing. A check cannot invent a cost. That one needs a person.

## Step 4: The same question again

In [7]:
print(MAIN_QUESTION)

analyst_run = run_analyst(MAIN_QUESTION)

print(f"\nAnswered in {analyst_run.seconds} seconds with {len(analyst_run.tool_log)} tool calls")

Which products and categories generated the most contribution profit last month, what changed, and how confident should I be in those numbers?

Answered in 16.8 seconds with 8 tool calls


> **If that cell reports an error**, run it again. It needs your OpenRouter key in the `.env` file.

A model does not answer the same way twice, and now and then an answer fails the verifier. In milestone 1 we stopped there. Here we do what a person would do: if the verifier rejects the answer, ask once more.

In [8]:
def is_verified(run):
    return run.verification is not None and run.verification.passed


if is_verified(analyst_run):
    print("The verifier accepted the first answer.")
else:
    print("The verifier rejected the first answer, so we ask once more.")
    analyst_run = run_analyst(MAIN_QUESTION)
    print("Second answer verified:", is_verified(analyst_run))

The verifier accepted the first answer.


In [9]:
show_answer(analyst_run.answer)

### In October 2026, Trail water bottle led contribution profit at $5,156.94 and Hydration led categories at $13,326.80. Apparel fell 37.8% from September, while Hydration rose 2.7%. Confidence is low because Camp stove has no product cost.
Top products: Trail water bottle $5,156.94, Headlamp $4,250.00, Rain jacket $3,841.00. Top categories: Hydration $13,326.80, Apparel $9,746.38, Camp $9,366.26. Apparel fell $5,914.87 (-37.8%) versus September and Hydration rose $344.80 (+2.7%). Camp's change cannot be calculated, and Camp stove (P007) is excluded from its total and from the product ranking because its product cost is missing (QF-004, high severity). Hydration and Apparel are complete. Duplicate-record and event-order flags are low severity. The 2026-09 Sun hat refund flag (QF-007) may understate that month's net sales.

| Figure | Value | Month |
|---|---|---|
| Trail water bottle, contribution profit, 2026-10 | $5,156.94 | 2026-10 |
| Headlamp, contribution profit, 2026-10 | $4,250.00 | 2026-10 |
| Rain jacket, contribution profit, 2026-10 | $3,841.00 | 2026-10 |
| Hydration, contribution profit, 2026-10 | $13,326.80 | 2026-10 |
| Apparel, contribution profit, 2026-10 | $9,746.38 | 2026-10 |
| Camp, contribution profit, 2026-10 (incomplete, excludes Camp stove) | $9,366.26 | 2026-10 |
| Apparel, contribution profit change, 2026-09 to 2026-10 | -$5,914.87 | 2026-10 |
| Apparel, contribution profit percentage change, 2026-09 to 2026-10 | -37.8% | 2026-10 |
| Hydration, contribution profit change, 2026-09 to 2026-10 | +$344.80 | 2026-10 |
| Hydration, contribution profit percentage change, 2026-09 to 2026-10 | +2.7% | 2026-10 |
| Camp, contribution profit change, 2026-09 to 2026-10 | unavailable | 2026-10 |

- **Caveat QF-004:** Camp stove (P007) has no product cost in 2026-10, so its gross profit and contribution profit are unavailable and it is left out of Camp's total and the product ranking. Camp's total is incomplete and its month-over-month change is null.
- **Caveat QF-002:** Trail water bottle had 9 duplicate records set aside; they are not in the figures.
- **Caveat QF-003:** Insulated flask had 12 duplicate records set aside; they are not in the figures.
- **Caveat QF-006:** Hydration pack had 1 record dated before its order; it is counted in the month of its own date.
- **Caveat QF-001:** September: Two-person tent had 1 duplicate record set aside; not in the figures.
- **Caveat QF-005:** September: Rain jacket had 1 record dated before its order; still counted.
- **Caveat QF-007:** September: Sun hat had 1 refund larger than its order line; still counted, so September net sales may be understated. This affects the Apparel comparison.

**Confidence: low.** QF-004 (high): Camp stove has no product cost in 2026-10, so Camp's total and change are incomplete. Low-severity duplicate and event-order flags in October and September. QF-007 (medium) in September may understate Apparel's September figure, which affects the Apparel change.

Read the caveats. They should mention the records that were set aside and the missing cost. The trail water bottle should be $5,156.94.

The verifier runs milestone 1's seven checks, and one new one that step 6 explains. One of the seven requires every finding the tool returned to appear in the caveats.

In [10]:
print("Verified:", analyst_run.verification.passed)
pd.DataFrame([check.model_dump() for check in analyst_run.verification.checks])

Verified: True


,name,passed,detail
0,figures_match_tools,True,11 figures match tool results
1,no_unsupported_numbers,True,Every number in the text comes from a tool result
2,sources_cited,True,"Cites ['metric_definitions', 'monthly_product_..."
3,definitions_stated,True,Every metric used has its definition and versi...
4,confidence_matches_rule,True,Answer says low; the rule gives low
5,flags_mentioned,True,All 7 returned flags are in the caveats
6,decline_is_clean,True,Not a declined answer
7,figures_listed,True,Amounts in the text have figures listed


## Step 5: Against the answer key

The answer key you corrected by hand in notebook 1, beside the calculated figures.

In [11]:
answer_key = load_result("answer_key", required_from="notebook 1")

check_key_against_calculated(answer_key)

,What,Answer key (by hand),Calculated (by the pipeline),Same?
0,#1 product: Trail water bottle,"$5,156.94","$5,156.94",yes
1,#2 product: Headlamp,"$4,250.00","$4,250.00",yes
2,#3 product: Rain jacket,"$3,841.00","$3,841.00",yes
3,Top category: Hydration,"$13,326.80","$13,326.80",yes
4,"Hydration, change since the month before",+$344.80,+$344.80,yes
5,"Apparel, change since the month before (the bi...","-$5,914.87","-$5,914.87",yes
6,"Camp, change since the month before (cannot be...",unavailable,unavailable,yes
7,Camp stove (its cost is missing),unavailable,unavailable,yes


| | Milestone 2 | Milestone 3 |
|---|---|---|
| Warnings | A list a person wrote | Findings from checks that run every time |
| A known duplicate | Counted, with a warning | Set aside, and still inspectable |
| A problem nobody listed | Missed | Found |
| Confidence | From the hand-written list | From real check results for the scope asked about |
| Who investigates a problem | A person, from scratch | An AI agent writes a first report; a person confirms |

## Step 6: The evaluation set

The evaluation set has grown from seven questions to ten. The three new ones ask about confidence and about records that were left out. Running the four "tiny" cases (E1, E2, E6, and the new E9) uses the analyst for real, about 20 model requests. It is off by default.

In [12]:
new_cases = [case for case in load_cases() if case.case_id in ("E8", "E9", "E10")]

pd.DataFrame([{"case": case.case_id, "question": case.question, "expected confidence": case.expected_confidence, "expected findings": case.expected_flags} for case in new_cases])

,case,question,expected confidence,expected findings
0,E8,What was the trail water bottle's contribution...,high,[QF-002]
1,E9,What were the sun hat's net sales in September...,medium,[QF-007]
2,E10,Were any records left out of the October 2026 ...,NaN,"[QF-002, QF-003]"


In [13]:
RUN_EVALUATION = False

if RUN_EVALUATION:
    scores = run_evaluation(run_analyst, load_cases(tiny=True))
    for score in scores:
        failed = [check.name for check in score.checks if not check.passed]
        print(f"{score.case_id}: {'passed' if score.passed else 'failed ' + str(failed)}")
        if score.error:
            print("   The model did not return an answer:", score.error)
    display(pd.Series(summarise(scores), name="passed").to_frame())
else:
    print("Evaluation not run. Set RUN_EVALUATION = True to run it.")

Evaluation not run. Set RUN_EVALUATION = True to run it.


The new questions taught us something when this lab was built. They ask about **one product**, and at first the analyst failed them about half the time. With more findings to report, the main answer had also grown very long. Five causes were found, by reading the raw replies:

| What went wrong | The fix |
|---|---|
| The question says "sun hat", but the quality tool only accepted `P011`. To find the id, the analyst fetched the whole month's findings, and then had to report all of them | The tool now also accepts a product's name |
| The analyst checked a wider scope than the question needed | One sentence in its instructions: use the narrowest scope that covers the answer |
| The main answer had grown to about 4,000 tokens, with up to 29 figures. At that size the model sometimes sent it malformed | The instructions now limit an answer to 15 figures and a short narrative |
| The model sometimes sent a list as text | The answer format now reads a list sent as text |
| The analyst sometimes wrote amounts in the text and left the `figures` list empty. Nothing caught it | A new, eighth verifier check, `figures_listed`, fails such an answer |

After those fixes, each question was run several times: the main question passed 5 of 6, E9 3 of 3, E10 2 of 3, and E8 1 of 3.

So the analyst is better, and still not perfectly reliable. That is the reason the verifier exists: when the model slips, code says so. In one E10 miss the analyst wrote that "21 records" were left out. It had added 9 and 12 itself, and the verifier caught a number that no tool returned.

None of this was found by reading code. It was found by asking new questions, measuring, and reading the raw replies.

## Step 7: From the lab to production

Everything you used from `retail_profit_stack` is the production code. This is where each notebook's work lives.

In [14]:
show_doc("LLD.md", "12. Where each part is built")

> **From `docs/LLD.md`: 12. Where each part is built**

The lab is six notebooks, run in order. Each line of notebook code does one thing, and each helper is shown before it is used.

| Notebook | What the learner does | Documents shown | This document's sections | Production code |
|---|---|---|---|---|
| 1. Where we are, and the problems by hand | Recalls the question and the answer key. Finds three problems in the records by hand, and works out what the trail water bottle's profit should be | BRD 1, 3, 5 | 3. Planted problems | `source_data.py` |
| 2. Checks in code | Writes two checks as SQL queries, tries each on a tiny clean table and a tiny broken one, then runs all five on the real records | HLD 5 and LLD 4 | 4, 10 | `quality/checks.py`, `quality/sql/` |
| 3. Quarantine, and the figures before and after | Sets the extra copies aside, reruns the calculation, compares the figures before and after, and runs the totals check | BRD 5 and LLD 5, 6 | 2, 5, 6 | `quality/quarantine.py`, `transform/pipeline.py` |
| 4. A to-do list and a helper, by hand | Asks one model call to explain all the findings at once. Then builds file tools, a to-do list, and a helper by hand, and investigates three findings | HLD 6 | 7, 8 | `quality/desk.py` |
| 5. A harness works the queue | Gives the whole desk to a Deep Agents agent, watches every step, scores its reports against the planted causes, and reads its handover note | HLD 7 and LLD 8 | 7, 8 | `quality/investigate.py`, `quality/desk.py` |
| 6. The same question, with real warnings | Asks the analyst the leader's question, sees the warnings come from the checks, runs the new evaluation questions, and maps each notebook to its production file | LLD 12 | 9, 11 | `store.py`, `tools.py`, `agent.py`, `verify.py`, `tests/` |

Notebooks 1 to 3 make no model requests. When the lab was built, notebook 4 made about 20, notebook 5 about 40 (about three US cents), and notebook 6 about 8. A full run cost about 10 US cents.

What the live runs showed when the lab was built: one model call named all seven causes correctly; the agent built by hand got its three right; and the harness got all seven right in each of five runs, across three different models.

## Step 8: Run the tests

Open a terminal in this milestone's folder (`src/m3_quality_trust_layer`) and run:

```
uv run pytest
```

The tests need no key and no network. Among other things they check that every check passes its clean cases and finds its planted ones, that kept plus quarantined equals the source, that each planted problem produces its finding, and that a broken report is left for a person.

To run the pipeline yourself from the terminal:

```
uv run python -m retail_profit_stack.transform.pipeline
```

> ### Your assignment
>
> 1. **Break a check and watch it get caught.** Open `retail_profit_stack/quality/sql/refund_exceeds_line.sql` and change `>` to `>=` on the last line. Run `uv run pytest`. Which case fails? Change it back.
> 2. **Add a check case.** In `evals/m3_check_cases.json`, add a case for `event_out_of_order` with a refund dated on the same day as its order. Should the check find it? Decide first, then run the tests.
> 3. **Change an outcome.** In `retail_profit_stack/quality/checks.py`, change the outcome of `refund_exceeds_line` to `quarantine` and rerun notebook 3. Which figure moves, and by how much? Is that a decision code should make on its own? Change it back.
> 4. **Stretch.** Write a sixth check: a shipment for a cancelled order. You need a SQL file, an entry in `CHECKS`, a cause in the rulebook, and two check cases.

> ### What you built
>
> 1. You found data problems by hand, and corrected the answer key.
> 2. You wrote checks as queries, and proved each one both ways.
> 3. You gave every check an outcome that code applies, quarantined the certain errors, and reconciled the totals.
> 4. You built a to-do list, file tools, and a helper by hand.
> 5. You used a harness to investigate a whole queue, and scored its work.
> 6. The leader got an answer whose warnings and confidence come from real checks.
>
> The lesson to carry forward: **code decides what happens to the data, and AI explains what code found.** A check gives the same verdict every time. A model is the right tool for reading seven messy cases and writing a report a person can act on, as long as its work is scored and it cannot touch the figures.
>
> Two things are still mocked. The records are still generated, and when the analyst shows evidence it is a sample of records. In Milestone 4 every figure becomes traceable to the exact records and the exact run that produced it.

**One last thing.** Open `milestone_3_recap.html`, in this milestone's folder, in your browser.